# RQ2: training conditions for the gender model (research plan B6, B4)

Trains the gender ViT on PETA under each condition, three seeds each, and scores every
run on the **person-disjoint** PETA test split, clean and under all 17 CCTV degradations.

| Condition | What it adds | Needs |
| :--- | :--- | :--- |
| `baseline` | the released stage-1 recipe | PETA |
| `degrade` | random CCTV degradations while training | PETA |
| `fda` | Fourier Domain Adaptation towards store frames | PETA + unlabelled store frames |
| `fda+degrade` | both | PETA + unlabelled store frames |

**Before you start**, put these in Google Drive under `MyDrive/IFAS_research/`:
- `PETA.zip`: the same file the stage-1 notebook asks you to upload.
- `fda_targets/` (optional): unlabelled frames from store cameras. Without it the two FDA
  conditions are skipped. Never use frames from clips that will be labelled for testing.

Then **Runtime → Change runtime type → GPU**, and **Run all**.

Each run takes roughly 20–30 minutes on a T4. Results are written to Drive as each run
finishes, and finished runs are skipped, so if Colab disconnects just run all again.

In [ ]:
# Settings
DRIVE_DIR  = '/content/drive/MyDrive/IFAS_research'
CONDITIONS = ['baseline', 'degrade', 'fda', 'fda+degrade']
SEEDS      = [0, 1, 2]
EPOCHS_B   = 5            # fine-tuning epochs (4 linear-probe epochs come first)
PACKS      = {}           # extra evaluation sets, e.g. {'mars': DRIVE_DIR + '/packs/mars'}
REPO       = 'https://github.com/Rishi0507/IFAS-Intelligent-Footfall-Analysis-System-for-Retail-Environments.git'
BRANCH     = 'main'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
os.makedirs(DRIVE_DIR, exist_ok=True)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

In [ ]:
# Code: the repository at BRANCH, with the pinned transformers version
if not os.path.exists('/content/ifas'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', BRANCH, REPO, '/content/ifas'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==5.6.2'], check=True)
os.chdir('/content/ifas'); sys.path.insert(0, '/content/ifas')
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
# Data: unzip PETA from Drive (once per session)
import zipfile, pathlib
PETA = '/content/peta'
if not os.path.exists(PETA):
    with zipfile.ZipFile(f'{DRIVE_DIR}/PETA.zip') as z:
        z.extractall(PETA)
labels = list(pathlib.Path(PETA).rglob('Label.txt'))
print(len(labels), 'PETA subsets found')
assert labels, 'No Label.txt found: is PETA.zip the raw PETA download?'

In [ ]:
# Split: built once and kept on Drive, so every run (and the age model) uses the same people
from src.research.peta import parse_peta, write_split_csv, read_split_csv
SPLIT = f'{DRIVE_DIR}/peta_split.csv'
if not os.path.exists(SPLIT):
    write_split_csv(parse_peta(PETA), SPLIT, seed=42)
for s in ('train', 'val', 'test'):
    print(s, len(read_split_csv(SPLIT, s)), 'images (person-disjoint)')

In [ ]:
# Train and evaluate every condition and seed; finished runs are skipped
from src.research.rq2 import train_run, fda_target_paths
RUNS = f'{DRIVE_DIR}/runs'
FDA_DIR = f'{DRIVE_DIR}/fda_targets'
have_fda = os.path.isdir(FDA_DIR) and len(fda_target_paths(FDA_DIR)) > 0
for cond in CONDITIONS:
    if 'fda' in cond and not have_fda:
        print(f'skipping {cond}: no frames in {FDA_DIR}')
        continue
    for seed in SEEDS:
        s = train_run(cond, seed, SPLIT, PETA, RUNS, fda_dir=FDA_DIR if 'fda' in cond else None,
                      eval_packs=PACKS, epochs_b=EPOCHS_B)
        print(f"== {cond} seed {seed}: clean test balanced accuracy {s['test']['clean']['balanced_accuracy']:.4f}")

In [ ]:
# How much did the old image-level split flatter the model? One baseline run on it.
s = train_run('baseline', 0, SPLIT, PETA, f'{DRIVE_DIR}/runs_image_split', epochs_b=EPOCHS_B,
              split_column='split_image')
print('image-level split, clean test balanced accuracy:', round(s['test']['clean']['balanced_accuracy'], 4))

In [ ]:
# Summary table; the JSON is what goes into paper/results/
import json
from src.research.rq2 import aggregate
agg = aggregate(RUNS)
with open(f'{DRIVE_DIR}/rq2_adaptation.json', 'w') as f:
    json.dump(agg, f, indent=1)
cols = ['clean', 'resolution:16', 'gaussian:3.0', 'motion:15', 'jpeg:10', 'truncation:0.5', 'occlusion:0.5']
print(f"{'condition':14}" + ''.join(f'{c:>20}' for c in cols))
for r in agg['rows']:
    print(f"{r['condition']:14}" + ''.join(
        f"{r['test'][c]['balanced_accuracy_mean']:>9.4f} +/- {r['test'][c]['balanced_accuracy_sd']:.3f}" for c in cols))
print()
print('Saved to', f'{DRIVE_DIR}/rq2_adaptation.json', '- send it back to be added to the paper.')